# Example 07 — FE Euler-Bernoulli Beam with Tanh Dry Friction

FRF near first bending resonance and spatial mode shape of a clamped-free Euler-Bernoulli beam with tanh dry friction at the midpoint node, using HB arc-length continuation. (Krack & Gross 2019, §5)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.linalg import eigh

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

# ── Patch FD step BEFORE importing hb_residual ──────────────────────────────
# The tanh friction uses eps=6e-7, so Q ~ 1e-8.
# The default _FD_STEP = sqrt(eps_machine) ≈ 1.5e-8 is too large relative to
# Q ~ 1e-8, causing poor Jacobian accuracy. Patching to 1.5e-15 so
# perturbations scale with Q rather than unity, restoring Newton convergence.
import nlvib.solvers.harmonic_balance as _hb_mod
_hb_mod._FD_STEP = 1.5e-15
# ─────────────────────────────────────────────────────────────────────────────

from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.nonlinearities.elements import tanh_dry_friction
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

print(f'_FD_STEP patched to: {_hb_mod._FD_STEP}')

In [ ]:
# --- System setup (parameters match MATLAB beam.mat / beam_tanhDryFriction_simple.m) ---
#
# MATLAB: len=2, height=0.05*len=0.1, thickness=3*height=0.3, E=185e9, rho=7830
#         BCs='clamped-free', n_nodes=9 → n_elements=8, n_dof=16
#         Friction: inode=4, muN=1.5, eps=6e-7 → c = 1/eps ≈ 1.67e6
#         Excitation: tip DOF 14, amplitude 0.2
#         H=7, omega sweep Om_s=370 → Om_e=110 rad/s

L_BEAM     = 2.0        # m
H_HEIGHT   = 0.1        # m  (0.05 * L_BEAM)
T_THICK    = 0.3        # m  (3 * H_HEIGHT)
E_MOD      = 185e9      # Pa
RHO        = 7830.0     # kg/m^3
N_ELEMENTS = 8          # 9 nodes → 8 elements → 16 free DOFs

I_AREA = H_HEIGHT**3 * T_THICK / 12   # cross-section second moment of area [m^4]
A_SECT = H_HEIGHT * T_THICK            # cross-section area [m^2]

MU_N       = 1.5        # friction limit force [N]
EPS        = 6e-7       # tanh regularisation parameter
C_TANH     = 1.0 / EPS  # ≈ 1.67e6

FORCE_AMP   = 0.2       # excitation amplitude [N]
N_HARMONICS = 7         # harmonics (matches MATLAB H=7)
OMEGA_LOW   = 110.0     # rad/s
OMEGA_HIGH  = 370.0     # rad/s

beam = FE_EulerBernoulliBeam(n_elements=N_ELEMENTS, L=L_BEAM, E=E_MOD,
                              I_area=I_AREA, rho=RHO, A=A_SECT, bc='clamped-free')

# Friction at node 3 (DOF 4 in Python 0-indexed = MATLAB DOF 4 in W vector)
FRICTION_DOF = beam.find_dof(3, 'w')
nl_element = tanh_dry_friction(f0=MU_N, c=C_TANH, dof_index=FRICTION_DOF)
beam.add_nonlinear_element(nl_element)

# Excitation at tip node 8, DOF 14
TIP_DOF = beam.find_dof(8, 'w')
excitation = {'dof': TIP_DOF, 'amplitude': FORCE_AMP, 'harmonic': 1}

n_dof = beam.n_dof
print(f'n_dof = {n_dof}  (expected 16)')
print(f'Friction DOF = {FRICTION_DOF}  (expected 4)')
print(f'Tip DOF      = {TIP_DOF}       (expected 14)')

K_dense = beam.K.toarray()
M_dense = beam.M.toarray()
eigvals, _ = eigh(K_dense, M_dense)
omega1_linear = float(np.sqrt(eigvals[0]))
print(f'Linear omega_1 = {omega1_linear:.3f} rad/s  (MATLAB: 123.341 rad/s)')

In [ ]:
# --- Initial solution + HB arc-length continuation ---
# The tanh with c=1/eps≈1.67e6 means Q~1e-8. Starting Newton from Q=0 can be
# difficult due to c-scaled Jacobian terms. Use modest initial Newton to get
# a starting point, then arc-length continue from OMEGA_LOW up to OMEGA_HIGH.

n_total = n_dof * (2 * N_HARMONICS + 1)
Q0 = np.zeros(n_total)
for _it in range(60):
    R, J = hb_residual(Q0, OMEGA_LOW, beam, N_HARMONICS, excitation)
    norm_R = np.linalg.norm(R)
    if norm_R < 1e-8:
        print(f'Newton converged at iteration {_it}: R = {norm_R:.3e}')
        break
    try:
        dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError:
        dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    step_norm = np.linalg.norm(dQ)
    if step_norm > 1e-6:
        dQ *= 1e-6 / step_norm
    Q0 += dQ
print(f'Initial residual at omega={OMEGA_LOW:.1f}: {np.linalg.norm(R):.3e}')

def residual_fn(Q, omega):
    return hb_residual(Q, omega, beam, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=5.0,
    ds_min=0.5,
    ds_max=15.0,
    max_steps=300,
    newton_tol=1e-6,
    lambda_min=OMEGA_LOW - 5.0,
    lambda_max=OMEGA_HIGH + 5.0,
    adapt_step=True,
)
result = ContinuationSolver().run(residual_fn, Q0, OMEGA_LOW, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract FRF (a_rms) at tip and save plots ---
#
# a_rms formula matches MATLAB beam_tanhDryFriction_simple.m:
#   Qtip = kron(eye(2H+1), T_tip) * Q_HB  → (2H+1, n_steps)
#   a_rms_HB = sqrt(sum(Qtip.^2)) / sqrt(2)
#
# Python equivalent: reshape Q to (n_steps, 2H+1, n_dof), extract TIP_DOF column

solutions  = result.solutions          # (n_steps, n_total + 1)
omegas     = solutions[:, -1]
Q_all      = solutions[:, :-1]         # (n_steps, n_dof*(2H+1))

# Reshape to (n_steps, 2H+1, n_dof) and extract tip DOF
Qtip  = Q_all.reshape(Q_all.shape[0], 2 * N_HARMONICS + 1, n_dof)[:, :, TIP_DOF]
a_rms = np.sqrt(np.sum(Qtip ** 2, axis=1)) / np.sqrt(2)

# Filter to comparison frequency range
mask      = (omegas >= OMEGA_LOW) & (omegas <= OMEGA_HIGH)
omegas_f  = omegas[mask]
a_rms_f   = a_rms[mask]

if len(a_rms_f) > 0:
    peak_idx      = int(np.argmax(a_rms_f))
    peak_amp      = float(a_rms_f[peak_idx])
    resonance_freq = float(omegas_f[peak_idx])
else:
    peak_amp = resonance_freq = float('nan')

output_dir = Path('..') / 'examples' / '07_beam_tanh_friction' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

# --- FRF plot: log y-scale, MATLAB-matching labels ---
fig_frf, ax_frf = plt.subplots(figsize=(8, 5))
ax_frf.semilogy(omegas_f, a_rms_f, 'b-', linewidth=2)
ax_frf.set_xlabel('excitation frequency')
ax_frf.set_ylabel('tip displacement amplitude')
ax_frf.set_xlim(OMEGA_LOW, OMEGA_HIGH)
ax_frf.set_title('Example 07 — Beam Tanh Friction FRF (tip a_rms)')
ax_frf.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.6)
ax_frf.axvline(resonance_freq, color='gray', linestyle=':', linewidth=0.8)
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf.png', dpi=150)
print('Saved frf.png')

# --- Mode shape at resonance peak ---
if len(omegas) > 0:
    global_peak_idx = int(np.argmax(a_rms))
    Q_peak = Q_all[global_peak_idx]   # shape: (n_total,)
    Q_peak_3d = Q_peak.reshape(2 * N_HARMONICS + 1, n_dof)  # (2H+1, n_dof)

    node_positions = [0.0]
    mode_shape_amp = [0.0]
    for node_i in range(1, N_ELEMENTS + 1):
        try:
            r_dof = beam.find_dof(node_i, 'w')
            # a_rms across all harmonics for this DOF
            q_harmonics = Q_peak_3d[:, r_dof]
            amp = float(np.sqrt(np.sum(q_harmonics ** 2)) / np.sqrt(2))
            node_positions.append(node_i * L_BEAM / N_ELEMENTS)
            mode_shape_amp.append(amp)
        except ValueError:
            pass

    node_positions_arr = np.array(node_positions)
    mode_shape_arr = np.array(mode_shape_amp)

    fig_mode, ax_mode = plt.subplots(figsize=(8, 4))
    ax_mode.plot(node_positions_arr, np.zeros_like(node_positions_arr),
                 'k--', linewidth=0.6, label='undeformed')
    ax_mode.plot(node_positions_arr, mode_shape_arr, 'o-', color='tab:green',
                 linewidth=1.5, label=f'mode shape at resonance ({resonance_freq:.1f} rad/s)')
    ax_mode.fill_between(node_positions_arr, mode_shape_arr, alpha=0.2, color='tab:green')
    ax_mode.set_xlabel('Position along beam (m)')
    ax_mode.set_ylabel('Transverse amplitude (m)')
    ax_mode.set_title('Example 07 — Mode Shape at Resonance Peak')
    ax_mode.legend()
    ax_mode.grid(True, alpha=0.3)
    fig_mode.tight_layout()
    fig_mode.savefig(output_dir / 'mode_shape.png', dpi=150)
    print('Saved mode_shape.png')

In [ ]:
# --- Display FRF inline ---
fig_frf

In [ ]:
# --- Display mode shape inline ---
fig_mode

In [ ]:
# --- Summary ---
print('=' * 55)
print('SUMMARY — Example 07: Beam Tanh Friction')
print('=' * 55)
print(f'  n_elements                   : {N_ELEMENTS}')
print(f'  L_beam                       : {L_BEAM} m')
print(f'  E_mod                        : {E_MOD:.3e} Pa')
print(f'  H (harmonics)                : {N_HARMONICS}')
print(f'  Friction muN / eps           : {MU_N} / {EPS}')
print(f'  Linear omega_1               : {omega1_linear:.3f} rad/s')
print(f'  Resonance frequency (HB)     : {resonance_freq:.2f} rad/s')
print(f'  Peak tip a_rms               : {peak_amp:.6e} m')
print(f'  _FD_STEP patch               : {_hb_mod._FD_STEP}')
print('=' * 55)